# 负面情感文档分类器（fastText 文本分类 + 推理）

本 Notebook 复刻「从关键词规则标注训练负面情感分类器」的流程，运行在华为云 ModelArts Notebook 中：

1. 数据准备：用负面/正面关键词规则给推文打标（与原流程一致）
2. 训练模型：fastText 文本分类（轻量、可复现，替代托管分类器训练）
3. 推理验证：对新文本实时分类（NEGATIVE / POSITIVE）
4. 上线（可选）：将模型封装为 ModelArts 在线推理端点或 FunctionGraph

> 原流程使用关键词匹配打标；生产环境建议用人工标注数据替换规则打标。


## 1. 依赖与数据准备

In [ ]:
# !pip install fasttext pandas

import pandas as pd

colnames = ['text']
tweets = pd.read_csv('../../data/tweet_telco.csv', encoding='utf-8', names=colnames, header=None)
print(tweets.shape)
tweets.head()

## 2. 关键词规则打标（与原流程一致）

In [ ]:
# 电信行业情感词典定义
# 业务场景说明：
# - 用于自动标注用户推文的情感倾向，支撑舆情监控、服务质量评估、投诉预警等业务
# - 电信行业负面词典：覆盖网络质量、服务态度、资费等典型投诉场景
# - 电信行业正面词典：覆盖套餐优惠、客服表扬、网络满意等场景
#
# 标注规则：
# - 命中负面词典 → NEGATIVE
# - 命中正面词典 → POSITIVE
# - 同时命中 → 负面优先（投诉场景需要更敏感的处理）

# 电信行业负面关键词（网络质量、服务态度、资费等）
neg_pat = r'(
    # 网络质量问题
    no[\s]?signal|no[\s]?service|no[\s]?coverage|bad[\s]?signal|poor[\s]?coverage|
    network[\s]?down|connection[\s]?lost|disconnected|can'?t[\s]?connect|
    slow[\s]?internet|slow[\s]?network|laggy|buffering|dropped[\s]?calls|
    # 服务质量问题
    terrible|awful|horrible|worst|useless|garbage|trash|
    angry|frustrated|annoyed|irritated|disgusted|disappointed|
    unhelpful|rude|unprofessional|poor[\s]?service|bad[\s]?service|
    cheated|scammed|ripped[\s]?off|overcharged|hidden[\s]?fee|
    # 投诉意向
    complain|complaint|will[\s]?cancel|want[\s]?to[\s]?quit|switch[\s]?carrier|
    never[\s]?again|not[\s]?recommend|stay[\s]?away|waste[\s]?of[\s]?time
)'

# 电信行业正面关键词（套餐优惠、客服表扬、网络满意等）
pos_pat = r'(
    # 网络满意度
    great[\s]?signal|excellent[\s]?coverage|fast[\s]?network|good[\s]?speed|
    love[\s]?this[\s]?network|best[\s]?carrier|amazing[\s]?service|
    # 客服表扬
    helpful|professional|friendly|great[\s]?customer[\s]?service|
    awesome|excellent|fantastic|perfect|wonderful|superb|
    thank[\s]?you|appreciate|pleased|satisfied|happy[\s]?with
    # 优惠活动
    great[\s]?deal|good[\s]?offer|discount|sale|bonus|free|
    recommend|worth[\s]?it|best[\s]?value|love[\s]?it
)'

tweets['match_negative'] = tweets['text'].str.contains(neg_pat, regex=True, case=False)
tweets['match_positive'] = tweets['text'].str.contains(pos_pat, regex=True, case=False)

# 冲突样本按负面优先（投诉预警需要更敏感）
tweets.loc[tweets['match_negative'] == True, 'label'] = 'NEGATIVE'
tweets.loc[tweets['match_positive'] == True, 'label'] = 'POSITIVE'
print("情感标注统计：")
print(tweets['label'].value_counts(dropna=False))
print(f"\n标注覆盖率: {tweets['label'].notna().sum() / len(tweets) * 100:.1f}%")

## 3. 构造训练集与测试集

In [ ]:
labeled = tweets[tweets['label'].notnull()].copy()
print(f"标注样本数: {len(labeled)}")

# fastText 需要 __label__ 前缀格式
labeled['ft_line'] = '__label__' + labeled['label'] + ' ' + labeled['text'].fillna('')
train_df = labeled.sample(frac=0.8, random_state=42)
test_df = labeled.drop(train_df.index)

train_df['ft_line'].to_csv('train_ft.txt', encoding='utf-8', index=False, header=False)
test_df['ft_line'].to_csv('test_ft.txt', encoding='utf-8', index=False, header=False)
print(f"训练 {len(train_df)} 条, 测试 {len(test_df)} 条")

## 4. 训练 fastText 分类器

In [ ]:
import fasttext

model = fasttext.train_supervised(
    input='train_ft.txt',
    epoch=25,
    lr=1.0,
    wordNgrams=2,
    minCount=1,
)
print("训练完成")

## 5. 测试集评估

In [ ]:
result = model.test('test_ft.txt')
print(f"样本数: {result[0]}, 精确率: {result[1]:.4f}, 召回率: {result[2]:.4f}")

## 6. 推理验证

In [ ]:
txt = "Great arena terrible wireless coverage here in vegas. why is coverage so bad? real bad at circus circus. MGM just ok."
pred = model.predict(txt, k=2)
print(f"文本: {txt}")
print(f"预测: {pred}")

## 7. 保存模型与上线（可选）

In [ ]:
model.save_model('negative_classifier.bin')
print("已保存: negative_classifier.bin")

# 可选：上传到 OBS
# !obsutil cp ./negative_classifier.bin obs://<bucket>/custom-nlp-pipeline/model/negative_classifier.bin

# 上线为 ModelArts 在线推理端点：在 ModelArts 控制台导入模型 + 部署，
# 或将本函数封装为 FunctionGraph（APIG 触发）提供 REST 分类接口。